In [1]:
import os
import pandas as pd

def analyze_dataset_structure(dataset_root):
    # Define paths
    overhead_dir = os.path.join(dataset_root, 'imagery', 'realsense_overhead')
    
    # 1. Analyze Metadata CSV files
    print("--- Metadata Analysis ---")
    csv_files = {
        'Nutrition Values': 'dish_nutrition_values.csv',
        'Dish Ingredients': 'dish_ingredients.csv',
        'Ingredients Meta': 'ingredients_metadata.csv'
    }
    
    for label, filename in csv_files.items():
        filepath = os.path.join(dataset_root, filename)
        if os.path.exists(filepath):
            df = pd.read_csv(filepath)
            print(f"{label} File: {filename}")
            print(f"Total Rows: {len(df)} | Columns: {list(df.columns)}\n")
        else:
            print(f"Missing File: {filename}\n")

    # 2. Analyze Image Directories (Overhead Only)
    print("--- Imagery Directory Structure (Overhead) ---")
    if os.path.exists(overhead_dir):
        # Get all subdirectories (which represent dish_ids)
        dish_folders = [d for d in os.listdir(overhead_dir) if os.path.isdir(os.path.join(overhead_dir, d))]
        print(f"Total Dish Directories (dish_id mapping): {len(dish_folders)}")
        
        # 3. Sample output to understand image naming and contents
        print("\nSampling 3 distinct dish directories to inspect image files:")
        for dish_id in dish_folders[:3]:
            dish_path = os.path.join(overhead_dir, dish_id)
            images = [f for f in os.listdir(dish_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
            
            print(f"Dish ID Folder: '{dish_id}' | Total Images: {len(images)}")
            if images:
                print(f"  -> Sample Image Names: {images[:3]}")
    else:
        print(f"Directory not found: {overhead_dir}")

# Expected usage in the notebook (assuming the dataset is in the current directory)
analyze_dataset_structure('/kaggle/input/nutrition5k-dataset')

--- Metadata Analysis ---
Nutrition Values File: dish_nutrition_values.csv
Total Rows: 4768 | Columns: ['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']

Dish Ingredients File: dish_ingredients.csv
Total Rows: 27225 | Columns: ['dish_id', 'ingr_id', 'ingr_name', 'grams', 'calories', 'fat', 'carb', 'protein']

Ingredients Meta File: ingredients_metadata.csv
Total Rows: 555 | Columns: ['ingr_name', 'ingr_id', 'cal/g', 'fat(g)', 'carb(g)', 'protein(g)']

--- Imagery Directory Structure (Overhead) ---
Total Dish Directories (dish_id mapping): 3490

Sampling 3 distinct dish directories to inspect image files:
Dish ID Folder: 'dish_1564588859' | Total Images: 3
  -> Sample Image Names: ['depth_raw.png', 'rgb.png', 'depth_color.png']
Dish ID Folder: 'dish_1561480439' | Total Images: 3
  -> Sample Image Names: ['depth_raw.png', 'rgb.png', 'depth_color.png']
Dish ID Folder: 'dish_1562615388' | Total Images: 3
  -> Sample Image Names: ['depth_raw.png', 'rgb.png', 'depth_color.png']


In [2]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

class Nutrition5kDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.overhead_dir = os.path.join(root_dir, 'imagery', 'realsense_overhead')
        
        # Load metadata
        nutr_df = pd.read_csv(os.path.join(root_dir, 'dish_nutrition_values.csv'))
        ingr_df = pd.read_csv(os.path.join(root_dir, 'dish_ingredients.csv'))
        meta_df = pd.read_csv(os.path.join(root_dir, 'ingredients_metadata.csv'))
        
        # Create a mapping for ingredient string to a fixed index (0 to 554)
        self.ingr_vocab = {row['ingr_id']: idx for idx, row in meta_df.iterrows()}
        self.num_ingredients = len(self.ingr_vocab)
        
        # Group ingredients by dish_id
        grouped_ingr = ingr_df.groupby('dish_id')['ingr_id'].apply(list).to_dict()
        
        self.valid_data = []
        
        # Filter and validate existing data
        for _, row in nutr_df.iterrows():
            dish_id = row['dish_id']
            img_path = os.path.join(self.overhead_dir, dish_id, 'rgb.png')
            
            # Only include if the image exists and ingredients are listed
            if os.path.exists(img_path) and dish_id in grouped_ingr:
                dish_ingr_list = grouped_ingr[dish_id]
                
                # Extract regression metrics: [Mass, Calories]
                metrics = [row['mass'], row['calories']]
                
                self.valid_data.append({
                    'dish_id': dish_id,
                    'img_path': img_path,
                    'ingredients': dish_ingr_list,
                    'metrics': metrics
                })
                
        print(f"Dataset initialized. Valid samples mapped: {len(self.valid_data)}")

    def __len__(self):
        return len(self.valid_data)

    def _get_multi_hot_ingredients(self, ingr_list):
        # Create a zero vector of size 555
        target = np.zeros(self.num_ingredients, dtype=np.float32)
        for ingr_id in ingr_list:
            if ingr_id in self.ingr_vocab:
                idx = self.ingr_vocab[ingr_id]
                target[idx] = 1.0
        return torch.tensor(target)

    def __getitem__(self, idx):
        item = self.valid_data[idx]
        
        # Load RGB image
        image = Image.open(item['img_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        # Prepare targets
        ingr_target = self._get_multi_hot_ingredients(item['ingredients'])
        # Fix: Use torch.float32 instead of np.float32
        metrics_target = torch.tensor(item['metrics'], dtype=torch.float32)
        
        return image, ingr_target, metrics_target
# Define transformations
# Note: RandomResizedCrop or heavy distortions are avoided to preserve volume estimation features
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test the Dataset implementation
dataset_root = '/kaggle/input/nutrition5k-dataset' # Replace with actual path
dataset = Nutrition5kDataset(root_dir=dataset_root, transform=train_transforms)

# Verify outputs
if len(dataset) > 0:
    sample_img, sample_ingr, sample_metrics = dataset[0]
    print(f"Image tensor shape: {sample_img.shape}")
    print(f"Ingredients target shape: {sample_ingr.shape} (Sum: {sample_ingr.sum().item()} active ingredients)")
    print(f"Metrics target shape: {sample_metrics.shape} -> Mass: {sample_metrics[0].item():.1f}g, Calories: {sample_metrics[1].item():.1f}kcal")

Dataset initialized. Valid samples mapped: 3244
Image tensor shape: torch.Size([3, 224, 224])
Ingredients target shape: torch.Size([555]) (Sum: 17.0 active ingredients)
Metrics target shape: torch.Size([2]) -> Mass: 193.0g, Calories: 300.8kcal


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import os

# 1. Define the Architecture (NutritionMTLModel)
class NutritionMTLModel(nn.Module):
    def __init__(self, num_ingredients, pretrained=True):
        super(NutritionMTLModel, self).__init__()
        
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.convnext_tiny(weights=weights)
        
        feature_dim = backbone.classifier[2].in_features
        
        self.feature_extractor = backbone.features
        self.avgpool = backbone.avgpool
        self.flatten = nn.Flatten()
        
        self._freeze_early_layers()

        self.ingredients_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Linear(512, num_ingredients) 
        )
        
        self.regression_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Linear(512, 2) 
        )

    def _freeze_early_layers(self):
        # Freeze initial stem
        for param in self.feature_extractor[0].parameters():
            param.requires_grad = False
            
        # Freeze stage 0 and stage 1
        for i in range(1, 4):
            for param in self.feature_extractor[i].parameters():
                param.requires_grad = False

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.avgpool(x)
        features = self.flatten(x)
        
        pred_ingredients = self.ingredients_head(features)
        pred_metrics = self.regression_head(features)
        
        return pred_ingredients, pred_metrics

print("Model architecture defined.")

# 2. Precise Dataset Splitting (Test: 500, Val: 500, Train: Rest)
total_size = len(dataset)
test_size = 500
val_size = 500
train_size = total_size - test_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# 3. Optimized DataLoaders for Kaggle Dual T4
batch_size = 64 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Data Split -> Train: {train_size} | Val: {val_size} | Test: {test_size}")

# 4. Model Initialization and Multi-GPU wrapper
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_ingredients = 555
model = NutritionMTLModel(num_ingredients=num_ingredients, pretrained=True)

if torch.cuda.device_count() > 1:
    print(f"Hardware optimization: Utilizing {torch.cuda.device_count()} GPUs via DataParallel.")
    model = nn.DataParallel(model)

model = model.to(device)

# 5. Loss Functions and Optimizer
criterion_ingredients = nn.BCEWithLogitsLoss()
criterion_metrics = nn.HuberLoss(delta=1.0)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)

scaler = GradScaler()

print("Setup complete. Ready for high-performance training.")

Model architecture defined.
Data Split -> Train: 2244 | Val: 500 | Test: 500
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 186MB/s]


Hardware optimization: Utilizing 2 GPUs via DataParallel.
Setup complete. Ready for high-performance training.


/tmp/ipykernel_23/1820192954.py:98: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [4]:
import time
import torch
import torch.nn as nn
import os
from torch.amp import autocast

# 1. Dynamically calculate pos_weight to combat extreme data sparsity
print("Calculating positive weights for sparse ingredients from training data...")
num_positives = torch.zeros(num_ingredients, device=device)
total_train_samples = 0

# Fast iteration without gradients to count active ingredients
with torch.no_grad():
    for _, target_ingr, _ in train_loader:
        num_positives += target_ingr.to(device).sum(dim=0)
        total_train_samples += target_ingr.size(0)

# Prevent division by zero for any ingredient that might be entirely absent
num_positives[num_positives == 0] = 1.0
num_negatives = total_train_samples - num_positives

# The weight applied to the positive class for each ingredient
pos_weight_vector = num_negatives / num_positives

# Redefine the classification loss with the calculated sparsity weights
criterion_ingredients = nn.BCEWithLogitsLoss(pos_weight=pos_weight_vector)
print("BCEWithLogitsLoss successfully updated with pos_weight.")

# 2. Training Configuration
num_epochs = 50  # Increased for deeper representation learning
alpha = 1.0    
beta = 0.01    

best_val_loss = float('inf')
save_dir = "./artifacts"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "best_nutrition_mtl_model.pth")

print(f"Starting Multi-Task Training Phase for {num_epochs} epochs...")

for epoch in range(num_epochs):
    start_time = time.time()
    
    # --- TRAINING PHASE ---
    model.train()
    train_loss, train_loss_ingr, train_loss_metr = 0.0, 0.0, 0.0
    
    for images, target_ingr, target_metrics in train_loader:
        images = images.to(device)
        target_ingr = target_ingr.to(device)
        target_metrics = target_metrics.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Suppress warnings by explicitly defining the device 'cuda'
        with autocast('cuda'):
            pred_ingr, pred_metrics = model(images)
            loss_ingr = criterion_ingredients(pred_ingr, target_ingr)
            loss_metr = criterion_metrics(pred_metrics, target_metrics)
            total_loss = (alpha * loss_ingr) + (beta * loss_metr)
            
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += total_loss.item()
        train_loss_ingr += loss_ingr.item()
        train_loss_metr += loss_metr.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_loss_ingr, val_loss_metr = 0.0, 0.0, 0.0
    
    with torch.no_grad():
        for images, target_ingr, target_metrics in val_loader:
            images = images.to(device)
            target_ingr = target_ingr.to(device)
            target_metrics = target_metrics.to(device)
            
            with autocast('cuda'):
                pred_ingr, pred_metrics = model(images)
                loss_ingr = criterion_ingredients(pred_ingr, target_ingr)
                loss_metr = criterion_metrics(pred_metrics, target_metrics)
                total_loss = (alpha * loss_ingr) + (beta * loss_metr)
                
            val_loss += total_loss.item()
            val_loss_ingr += loss_ingr.item()
            val_loss_metr += loss_metr.item()
            
    avg_val_loss = val_loss / len(val_loader)
    epoch_time = time.time() - start_time
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {epoch_time:.1f}s")
    print(f"  Train Loss: {avg_train_loss:.4f} (Ingr: {train_loss_ingr/len(train_loader):.4f}, Metr: {train_loss_metr/len(train_loader):.4f})")
    print(f"  Val Loss:   {avg_val_loss:.4f} (Ingr: {val_loss_ingr/len(val_loader):.4f}, Metr: {val_loss_metr/len(val_loader):.4f})")
    
    # Checkpointing
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state_dict, best_model_path)
        print(f"  -> Best model saved at validation loss: {best_val_loss:.4f}")

print("Training Phase Completed.")

Calculating positive weights for sparse ingredients from training data...
BCEWithLogitsLoss successfully updated with pos_weight.
Starting Multi-Task Training Phase for 50 epochs...
Epoch [1/50] | Time: 276.7s
  Train Loss: 3.1175 (Ingr: 0.8167, Metr: 230.0794)
  Val Loss:   2.6962 (Ingr: 0.5987, Metr: 209.7564)
  -> Best model saved at validation loss: 2.6962
Epoch [2/50] | Time: 273.9s
  Train Loss: 2.3905 (Ingr: 0.6497, Metr: 174.0825)
  Val Loss:   2.0195 (Ingr: 0.5551, Metr: 146.4349)
  -> Best model saved at validation loss: 2.0195
Epoch [3/50] | Time: 281.3s
  Train Loss: 1.8323 (Ingr: 0.5894, Metr: 124.2951)
  Val Loss:   1.4009 (Ingr: 0.5339, Metr: 86.6994)
  -> Best model saved at validation loss: 1.4009
Epoch [4/50] | Time: 292.8s
  Train Loss: 1.3934 (Ingr: 0.5462, Metr: 84.7210)
  Val Loss:   1.2763 (Ingr: 0.5447, Metr: 73.1641)
  -> Best model saved at validation loss: 1.2763
Epoch [5/50] | Time: 287.3s
  Train Loss: 1.2731 (Ingr: 0.5269, Metr: 74.6168)
  Val Loss:   1.23